This code detects the location of starting point of an office;

Input: List of offices + Corresponding offices 

=> Output: List of offices with coordinate information

In [118]:
import pandas as pd
import os
import cv2
import numpy as np
import json

In [119]:
def Detect_Office(Json,Office):

    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if (Office[0] == d['inferText'][0]) or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

def Detect_Office2(Json,Office):
    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if ( d['inferText'][0] == '〇') or( d['inferText'][0] == '○') or ( d['inferText'][0] == '0')or ( d['inferText'][0] == 'O') or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

def Detect_Office3(Json,Office):
    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if ( d['inferText'][0] == '〇') or( d['inferText'][0] == '○') or ( d['inferText'][0] == '0')or ( d['inferText'][0] == 'O')]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)
    
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(Year,Page):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
    with open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg",'rb') as f:
         file_data = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

def Clova2(Year,Page):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg", "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    HH,WW=img.shape[:2]
    cropped=img[0:200,0:WW]
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
    cv2.imwrite(temp_path+"Temp.jpg",cropped)
    
    with open(temp_path+"Temp.jpg",'rb') as f:
        file_data = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

In [120]:
def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation)
                      for im in im_list]
    return cv2.hconcat(im_list_resize)

def Check(df):
    for n in range(0,len(df)):
        #Extract key info of office
        Row  = df.iloc[n]

        ###Collect Location information###
        Dept=Row["Dept"]
        Office=Row["Office"]
        print('['+str(n)+',"'+Dept+'","'+Office+'"]')
        try:        
            StrPage=dta[Year][Dept][Office]['Starting_Page']
            EndPage=dta[Year][Dept][Office]['Ending_Page']

            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb')
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            oldimg=img.copy()[0:200,0:WW]

            for Page in range(StrPage+1,EndPage+1):
                path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
                stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb') 
                bytes = bytearray(stream.read())
                numpyarray = np.asarray(bytes, dtype=np.uint8)
                newimg = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)[0:200,0:WW]
                oldimg=np.concatenate((newimg,oldimg), axis=1)
            oldimg=cv2.copyMakeBorder(oldimg,20,20,20,20,cv2.BORDER_CONSTANT)

            StrLoc=int(dta[Year][Dept][Office]['Office_X1'])
            EndLoc=int(dta[Year][Dept][Office]['Office_X2'])
            StrPage=int(dta[Year][Dept][Office]['Starting_Page'])
            EndPage=int(dta[Year][Dept][Office]['Ending_Page'])

            #Start#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+"Page"+"{:03d}".format(StrPage)+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(StrLoc,0),(StrLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            #End#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(EndPage)+"\\"+"Page"+"{:03d}".format(EndPage)+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(EndLoc,0),(EndLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            cv2.imshow(Dept+Office,oldimg)
            cv2.waitKey(0)
        except:
            continue
    cv2.destroyAllWindows()
    cv2.waitKey(0)

In [121]:
def Show(n,StrLoc):
    Row=df.iloc[n]
    Page=Row['Page']
    
    path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg",'rb')
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    
    HH,WW=img.shape[:2]
    cv2.line(img,(StrLoc,0),(StrLoc,HH),(225,0,0),2)
    cv2.imshow('Projection',img)
    cv2.waitKey(0)

#Step 1

Fill in years to refer.
Remember to keep it as a string. NOT as float.

In [55]:
Year='1939'
Showa='14'
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
os.chdir(path)
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1).dropna()

file_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+Year+'\\DataFrame.json'
with open(file_path, encoding="utf-8") as f:
    dta = json.loads(f.read())

#Step 2

The following code will extract the location of offices.

In [59]:
n=0
FailedList=[]
#Extract key info of office
Row  = df.iloc[n]
ExRow= df.iloc[n-1]

Page=int(Row["Page"])
Office=Row["Office"]
Dept=Row['Dept']


###Insert Starting page information to motherframe###
dta[Year][Dept][Office]={}
dta[Year][Dept][Office]["Starting_Page"]=Page
print(dta[Year][Dept][Office])

###Collect Location information###
##Read image for first page##
img=cv2.imread("Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
#Convert to json via CLOVA

Json=Clova(Year,Page)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
if XCoord_Unit==None:
    #Add to motherframe
    dta[str(Year)][Dept][Office]["Office_X1"]='NA'
    print(Office+ 'failed')
    FailedList.append(list((n,Dept,Office)))
else:
    dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
    HH=img.shape[:2][0]
    print(Office+ 'success: at row'+str(n))

{'Starting_Page': 2}
秘書課failed


In [61]:
#Test code| Version 2#
#Show Working office#
for n in range(1,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Insert Starting page information to motherframe###
    dta[Year][Dept][Office]={}
    dta[Year][Dept][Office]["Starting_Page"]=Page
    print(dta[Year][Dept][Office])

    ###Collect Location information###
    ##Read image for first page##
    img=cv2.imread("Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
    #Convert to json via CLOVA
    try:
        Json=Clova(Year,Page)
    except:
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["Office_X1"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        HH=img.shape[:2][0]
        print(Office+ 'success: at row'+str(n))

{'Starting_Page': 2}
文書課success: at row1
{'Starting_Page': 5}
人事課success: at row2
{'Starting_Page': 6}
吏務課success: at row3
{'Starting_Page': 7}
議案課success: at row4
{'Starting_Page': 8}
企画課failed
{'Starting_Page': 9}
都市計画課success: at row6
{'Starting_Page': 12}
統計課success: at row7
{'Starting_Page': 15}
市務監察課failed
{'Starting_Page': 16}
区務監察課failed
{'Starting_Page': 17}
会計課success: at row10
{'Starting_Page': 20}
主計課success: at row11
{'Starting_Page': 21}
主税課success: at row12
{'Starting_Page': 28}
公債課success: at row13
{'Starting_Page': 89}
購買課failed
{'Starting_Page': 32}
用品課success: at row15
{'Starting_Page': 35}
地理課success: at row16
{'Starting_Page': 40}
第一課工営課failed
{'Starting_Page': 43}
第二課工営課failed
{'Starting_Page': 44}
庶務課success: at row19
{'Starting_Page': 45}
区町課success: at row20
{'Starting_Page': 47}
社会教育課failed
{'Starting_Page': 52}
體力課success: at row22
{'Starting_Page': 53}
公園課failed
{'Starting_Page': 60}
防衛課success: at row24
{'Starting_Page': 62}
観光課success: at row25
{'Starting_

Fix the department-office pair with errors

First re-run the list of failed (pairs which was rejected), with a stricter OCR.

In [62]:
FailedList2=[]
for Office in FailedList:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    Json=Clova2(Year,Page)
    
    try:
        XCoord_Unit=Detect_Office2(Json,Office)
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row '+str(n))
        img=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        #cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,300), (255,0,0), 2)
        #cv2.imshow('pic',img)
        #cv2.waitKey(0)
    except:
        FailedList2.append(list((n,Dept,Office)))
        continue

秘書課success: at row 0
市務監察課success: at row 8
区務監察課success: at row 9
第一課工営課success: at row 17
第二課工営課success: at row 18
社会教育課success: at row 21
公園課success: at row 23
計画課success: at row 26
学校職員課success: at row 29
学校体育課success: at row 33
第一作業課success: at row 43
第二作業課success: at row 44
道路建設課success: at row 63
事業普及課success: at row 84


In [65]:
FailedList2

[[5, '総務局', '企画課'],
 [14, '財務局', '購買課'],
 [50, '療養所', '管理課'],
 [54, '経済局', '貿易課'],
 [83, '運輸部', '業務課'],
 [92, '病院', '監務課'],
 [93, '病院', '診療所']]

In [68]:
#Fixing Failed Offices
#Step1: Check for simple page assignment error
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
for n in range(0,len(FailedList2)):
    Dept=FailedList2[n][1]
    Office=FailedList2[n][2]
    Page=df['Page'][(df['Office']==Office) & (df['Dept']==Dept)].tolist()[0]
    print('['+str(n)+',"'+Dept+'","'+Office+'","'+str(Page)+'"]')
    image=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
    cv2.imshow('Image',image)
    cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.waitKey(0)

[0,"総務局","企画課","8"]
[1,"財務局","購買課","89"]
[2,"療養所","管理課","175"]
[3,"経済局","貿易課","185"]
[4,"運輸部","業務課","286"]
[5,"病院","監務課","337"]
[6,"病院","診療所","339"]


-1

In [127]:
for k in range(0,len(FailedList2)):    
    n,Office,Dept=FailedList2[k][0],FailedList2[k][2],FailedList2[k][1]
    
    print(Dept,Office,n)
    dta[Year][Dept][Office]['Office_X1']=0

総務局 企画課 5
財務局 購買課 14
療養所 管理課 50
経済局 貿易課 54
運輸部 業務課 83
病院 監務課 92
病院 診療所 93


#Step 3

See how much errors were observed in the first trial.

In [14]:
MissAlignment=[["Admin","秘書課","1"],
               ['総務局','統計課','9'],
               ['財務局','購買課','89'],
              ["病院","監務課","337"],
              ["病院","診療所","339"]]

PageCut=[["経済局","貿易課","185"],
        ["運輸部","業務課","286"]]

Type1=[["総務局","企画課","8"]]

#Step 4

Check for errors.
Manually type in the index, department name, and office name of erroneous department-office pair to a seperate list.

In [128]:
FailRate=len(FailedList2)/len(df)
if FailRate<0.2:
    print('Fantastic!! Success Rate is '+str(1-FailRate)+' '+str(len(FailedList2)))
else:
    print('残念...Success Rate is '+str(1-FailRate)+' '+len(FailedList2))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index_col=False)
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate
display(DF)
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index=False)

Fantastic!! Success Rate is 0.9278350515463918 7


,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Year,WritePage,Page,PageFin,Office,OfficeFin,Unit,UnitFin,Position,PositionFin,Data
0,0,0,0,0,1934,1,0.985714286,.,0.9,.,NaN,.,0.08,.,.
1,1,1,1,1,1935,1,1,.,0.984375,.,NaN,.,-0.09375,.,.
2,2,2,2,2,1936,0.235294118,.,.,0.235294118,.,NaN,.,.,.,.
3,3,3,3,3,1937,1,1,.,0.865671642,.,NaN,.,.,.,.
4,4,4,4,4,1938,1,1,.,1,.,0.988095238,.,0.98816568,.,0.619957537
5,5,5,5,5,1939,1,1.0,.,0.927835,.,.,.,.,.,.
6,6,6,6,6,1940,1,.,.,0.991803279,.,NaN,.,.,.,.
7,7,7,7,7,1941,1,1,.,.,.,NaN,.,.,.,.
8,8,8,8,8,1942,1,1,.,.,.,NaN,.,.,.,.
9,9,9,9,9,1943,1,.,.,.,.,.,.,.,.,.


In [73]:
Check(df)

[0,"Admin","秘書課"]
[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"監査部","市務監察課"]
[9,"監査部","区務監察課"]
[10,"財務局","会計課"]
[11,"財務局","主計課"]
[12,"財務局","主税課"]
[13,"財務局","公債課"]
[14,"財務局","購買課"]
[15,"財務局","用品課"]
[16,"財務局","地理課"]
[17,"建築部","第一課工営課"]
[18,"建築部","第二課工営課"]
[19,"市民局","庶務課"]
[20,"市民局","区町課"]
[21,"市民局","社会教育課"]
[22,"市民局","體力課"]
[23,"市民局","公園課"]
[24,"市民局","防衛課"]
[25,"市民局","観光課"]
[26,"記念事業部","計画課"]
[27,"記念事業部","事業課"]
[28,"教育局","庶務課"]
[29,"教育局","学校職員課"]
[30,"教育局","学務課"]
[31,"教育局","視学課"]
[32,"教育局","青年教育課"]
[33,"教育局","学校体育課"]
[34,"教育局","教育研究所"]
[35,"厚生局","庶務課"]
[36,"厚生局","軍事援護課"]
[37,"厚生局","児童課"]
[38,"厚生局","保護課"]
[39,"厚生局","福利課"]
[40,"厚生局","衛生課"]
[41,"厚生局","防疫課"]
[42,"清掃部","監理課"]
[43,"清掃部","第一作業課"]
[44,"清掃部","第二作業課"]
[45,"養育院","庶務課"]
[46,"養育院","監護課"]
[47,"養育院","医務課"]
[48,"養育院","会計課"]
[49,"養育院","分院及学校"]
[50,"療養所","管理課"]
[51,"療養所","監務課"]
[52,"経済局","庶務課"]
[53,"経済局","商工課"]
[54,"経済局","貿易課"]
[55,"経済局","金融課"]
[56,"経済局","農漁課"]
[57,"

Type in the department-offices with errors.


In [78]:
Type1=[[0,"Admin","秘書課"],
      [11,"財務局","主計課"],
      [12,"財務局","主税課"],
      [24,"市民局","防衛課"],
      [31,"教育局","視学課"],
       [32,"教育局","青年教育課"],
      [34,"教育局","教育研究所"],
      [36,"厚生局","軍事援護課"],
      [49,"養育院","分院及学校"],
      [50,"療養所","管理課"],
      [60,"中央卸売市場","副収入役室"],
      [65,"土木局","治水工事課"],
      [67,"港湾局","技術課"],
      [68,"港湾局","港務所"],
      [69,"港湾局","臨時飛行場建設事務所"],
      [75,"水道局","下水課"],
      [85,"運輸部","車両課"],
       [87,"運輸部","交通統制調査課"],
       [89,"電燈部","配線課"],
       [94,"電気研究所","庶務課"],
       [95,"電気研究所","試験課"]]+FailedList2

Open FailedList_2 and see if there are any other errors left.

If there are, fix it manually by guessing the starting x axis using 'Show(n,StrLoc)'

When you find the exact starting location, update the dataframe.

dta[Year][Dept][Office]={'Starting_Page': ,
                          'Office_X1': ,
                          'Ending_Page': ,
                          'Office_X2': ,
                          'Page_Range': []}

#Step 5

Do the same thing with the department-office pair which was wrongly accepted in the test.

In [129]:
Type1_2=[]
for Office in Type1:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    Json=Clova2(Year,Page)
    
    try:
        XCoord_Unit=Detect_Office3(Json,Office)
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row '+str(n))
        img=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        #cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,300), (255,0,0), 2)
        #cv2.imshow('pic',img)
        #cv2.waitKey(0)
    except:
        Type1_2.append(list((n,Dept,Office)))
        continue

秘書課success: at row 0
主税課success: at row 12
防衛課success: at row 24
青年教育課success: at row 32
教育研究所success: at row 34
管理課success: at row 50
副収入役室success: at row 60
治水工事課success: at row 65
技術課success: at row 67
港務所success: at row 68
臨時飛行場建設事務所success: at row 69
下水課success: at row 75
車両課success: at row 85
交通統制調査課success: at row 87
管理課success: at row 50


#Step 6

Check the entire dataframe to confirm that the lines have been drawn at the correct place.

If there are errors, add the pair into the Type1 Error list and re-run step 5.

In [107]:
Check(df)

[0,"Admin","秘書課"]
[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"監査部","市務監察課"]
[9,"監査部","区務監察課"]
[10,"財務局","会計課"]
[11,"財務局","主計課"]
[12,"財務局","主税課"]
[13,"財務局","公債課"]
[14,"財務局","購買課"]
[15,"財務局","用品課"]
[16,"財務局","地理課"]
[17,"建築部","第一課工営課"]
[18,"建築部","第二課工営課"]
[19,"市民局","庶務課"]
[20,"市民局","区町課"]
[21,"市民局","社会教育課"]
[22,"市民局","體力課"]
[23,"市民局","公園課"]
[24,"市民局","防衛課"]
[25,"市民局","観光課"]
[26,"記念事業部","計画課"]
[27,"記念事業部","事業課"]
[28,"教育局","庶務課"]
[29,"教育局","学校職員課"]
[30,"教育局","学務課"]
[31,"教育局","視学課"]
[32,"教育局","青年教育課"]
[33,"教育局","学校体育課"]
[34,"教育局","教育研究所"]
[35,"厚生局","庶務課"]
[36,"厚生局","軍事援護課"]
[37,"厚生局","児童課"]
[38,"厚生局","保護課"]
[39,"厚生局","福利課"]
[40,"厚生局","衛生課"]
[41,"厚生局","防疫課"]
[42,"清掃部","監理課"]
[43,"清掃部","第一作業課"]
[44,"清掃部","第二作業課"]
[45,"養育院","庶務課"]
[46,"養育院","監護課"]
[47,"養育院","医務課"]
[48,"養育院","会計課"]
[49,"養育院","分院及学校"]
[50,"療養所","管理課"]
[51,"療養所","監務課"]
[52,"経済局","庶務課"]
[53,"経済局","商工課"]
[54,"経済局","貿易課"]
[55,"経済局","金融課"]
[56,"経済局","農漁課"]
[57,"

In [85]:
Fin=[[0,"Admin","秘書課"],
         [7,"総務局","統計課"],
        [1,"総務局","文書課"]]
len(Fin)

3

In [131]:
k=0
n,Office,Dept=Fin[k][0],Fin[k][2],Fin[k][1]
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Show(n,460)
dta[Year][Dept][Office]['Office_X1']=460


In [133]:
k=1
n,Office,Dept=Fin[k][0],Fin[k][2],Fin[k][1]
ExRow=df.iloc[n-1]
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Show(n,340)
dta[Year][Dept][Office]['Office_X1']=340
dta[Year][ExDept][ExOffice]['Office_X2']=340

TypeError: 'int' object does not support item assignment

In [125]:
k=2
n,Office,Dept=Fin[k][0],Fin[k][2],Fin[k][1]
ExRow=ExRow=df[n-1]
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Show(n,20)
dta[Year][Dept][Office]['Office_X1']=20
dta[Year][ExDept][ExOffice]['Office_X2']=20

In [136]:
Check(df)

[0,"Admin","秘書課"]
[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"監査部","市務監察課"]
[9,"監査部","区務監察課"]
[10,"財務局","会計課"]
[11,"財務局","主計課"]
[12,"財務局","主税課"]
[13,"財務局","公債課"]
[14,"財務局","購買課"]
[15,"財務局","用品課"]
[16,"財務局","地理課"]
[17,"建築部","第一課工営課"]
[18,"建築部","第二課工営課"]
[19,"市民局","庶務課"]
[20,"市民局","区町課"]
[21,"市民局","社会教育課"]
[22,"市民局","體力課"]
[23,"市民局","公園課"]
[24,"市民局","防衛課"]
[25,"市民局","観光課"]
[26,"記念事業部","計画課"]
[27,"記念事業部","事業課"]
[28,"教育局","庶務課"]
[29,"教育局","学校職員課"]
[30,"教育局","学務課"]
[31,"教育局","視学課"]
[32,"教育局","青年教育課"]
[33,"教育局","学校体育課"]
[34,"教育局","教育研究所"]
[35,"厚生局","庶務課"]
[36,"厚生局","軍事援護課"]
[37,"厚生局","児童課"]
[38,"厚生局","保護課"]
[39,"厚生局","福利課"]
[40,"厚生局","衛生課"]
[41,"厚生局","防疫課"]
[42,"清掃部","監理課"]
[43,"清掃部","第一作業課"]
[44,"清掃部","第二作業課"]
[45,"養育院","庶務課"]
[46,"養育院","監護課"]
[47,"養育院","医務課"]
[48,"養育院","会計課"]
[49,"養育院","分院及学校"]
[50,"療養所","管理課"]
[51,"療養所","監務課"]
[52,"経済局","庶務課"]
[53,"経済局","商工課"]
[54,"経済局","貿易課"]
[55,"経済局","金融課"]
[56,"経済局","農漁課"]
[57,"

In [137]:
EndList=[[1,"総務局","文書課"],
        [8,"監査部","市務監察課"],
        [12,"財務局","主税課"],
        [26,"記念事業部","計画課"],
        [35,"厚生局","庶務課"],
         [36,"厚生局","軍事援護課"],
        [64,"土木局","河川課"],
        [96,"電気研究所","研究課"]]

In [140]:
for Office in EndList:
    n,Dept,Office=Office[0],Office[1],Office[2]
    EndLoc=dta[Year][Dept][Office]['Office_X1']
    EndPage=dta[Year][Dept][Office]['Ending_Page']
    
    ExRow=df.iloc[n-1]
    ExDept,ExOffice=ExRow['Dept'],ExRow['Office']
    dta[Year][ExDept][ExOffice]=EndPage

In [141]:
Check(df)

[0,"Admin","秘書課"]
[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"監査部","市務監察課"]
[9,"監査部","区務監察課"]
[10,"財務局","会計課"]
[11,"財務局","主計課"]
[12,"財務局","主税課"]
[13,"財務局","公債課"]
[14,"財務局","購買課"]
[15,"財務局","用品課"]
[16,"財務局","地理課"]
[17,"建築部","第一課工営課"]
[18,"建築部","第二課工営課"]
[19,"市民局","庶務課"]
[20,"市民局","区町課"]
[21,"市民局","社会教育課"]
[22,"市民局","體力課"]
[23,"市民局","公園課"]
[24,"市民局","防衛課"]
[25,"市民局","観光課"]
[26,"記念事業部","計画課"]
[27,"記念事業部","事業課"]
[28,"教育局","庶務課"]
[29,"教育局","学校職員課"]
[30,"教育局","学務課"]
[31,"教育局","視学課"]
[32,"教育局","青年教育課"]
[33,"教育局","学校体育課"]
[34,"教育局","教育研究所"]
[35,"厚生局","庶務課"]
[36,"厚生局","軍事援護課"]
[37,"厚生局","児童課"]
[38,"厚生局","保護課"]
[39,"厚生局","福利課"]
[40,"厚生局","衛生課"]
[41,"厚生局","防疫課"]
[42,"清掃部","監理課"]
[43,"清掃部","第一作業課"]
[44,"清掃部","第二作業課"]
[45,"養育院","庶務課"]
[46,"養育院","監護課"]
[47,"養育院","医務課"]
[48,"養育院","会計課"]
[49,"養育院","分院及学校"]
[50,"療養所","管理課"]
[51,"療養所","監務課"]
[52,"経済局","庶務課"]
[53,"経済局","商工課"]
[54,"経済局","貿易課"]
[55,"経済局","金融課"]
[56,"経済局","農漁課"]
[57,"

In [142]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+str(Year)+'\\'
with open(save_path+"DataFrame_Office.json", "w") as outfile:
    outfile.write(json_object)